# SneakyToS — Version Pairs and Surface-Diff Baselines

This notebook is the second stage of the SneakyToS pipeline.

## Goals

1. Load the historical Zoom Privacy Policy dataset created in Notebook 01.
2. Collapse exact duplicate policy texts using SHA-256 hashes.
3. Preserve duplicate-snapshot metadata for auditability.
4. Create chronologically consecutive **unique-policy** version pairs.
5. Compute transparent surface-level text-difference baselines.
6. Inspect the largest and smallest policy updates.
7. Save the version-pair dataset for clause segmentation and semantic analysis.

> This notebook deliberately does **not** attempt to decide whether a change is a rights expansion. These are baseline measurements of textual change only.


## 1. Imports

In [2]:
from pathlib import Path
from difflib import SequenceMatcher, unified_diff
import re
import pandas as pd
import numpy as np


## 2. Project paths

In [3]:
cwd = Path.cwd()

if (cwd / "data" / "processed").exists():
    PROJECT_ROOT = cwd
elif (cwd.parent / "data" / "processed").exists():
    PROJECT_ROOT = cwd.parent
else:
    raise FileNotFoundError(
        "Could not locate the project root containing data/processed."
    )

PROCESSED_PATH = PROJECT_ROOT / "data" / "processed"
VERSIONS_PATH = PROCESSED_PATH / "zoom_policy_versions.pkl"

if not VERSIONS_PATH.exists():
    raise FileNotFoundError(
        f"Notebook 01 output not found: {VERSIONS_PATH}"
    )

print("Project root:", PROJECT_ROOT)
print("Input dataset:", VERSIONS_PATH)


Project root: c:\Users\arjun\Desktop\SneakyToS
Input dataset: c:\Users\arjun\Desktop\SneakyToS\data\processed\zoom_policy_versions.pkl


## 3. Load Zoom policy snapshots

In [4]:
zoom_versions_df = pd.read_pickle(VERSIONS_PATH).copy()

required_columns = {
    "company",
    "commit_hash",
    "timestamp",
    "commit_message",
    "text",
    "text_hash",
    "char_count",
    "word_count",
}

missing_columns = required_columns - set(zoom_versions_df.columns)

if missing_columns:
    raise ValueError(
        f"Missing expected columns: {sorted(missing_columns)}"
    )

zoom_versions_df["timestamp"] = pd.to_datetime(
    zoom_versions_df["timestamp"],
    utc=True,
)

zoom_versions_df = (
    zoom_versions_df
    .sort_values("timestamp")
    .reset_index(drop=True)
)

print("Loaded snapshots:", len(zoom_versions_df))
print("Unique policy texts:", zoom_versions_df["text_hash"].nunique())
print()
display(
    zoom_versions_df[
        [
            "timestamp",
            "commit_hash",
            "commit_message",
            "word_count",
            "char_count",
            "text_hash",
        ]
    ]
)


Loaded snapshots: 30
Unique policy texts: 28



,timestamp,commit_hash,commit_message,word_count,char_count,text_hash
0,2021-08-16 08:55:23+00:00,f5197ca248b099f54512cb518181241af1642749,Start tracking Zoom Privacy Policy,4679,31458,4b2e66fe8b0ba75a7d2510623c618fba549d2fde360080...
1,2021-10-07 21:14:03+00:00,4788e3698c43bc99597d7fd2836b138ef8145e85,Update Zoom Privacy Policy,4705,31710,330b2a9d5e468d54cec04499bcf135d0f34ce223923db0...
2,2021-11-03 05:10:18+00:00,4fbd6abb19ff42c3a828c95327248576a4966e0a,Refilter Zoom Privacy Policy,4829,32622,1c3199181727797b4b323da4d20c197d3175d80ac55985...
3,2021-11-23 10:17:24+00:00,7669087fd9438400efc97cc939a9a061588e6da2,Update Zoom Privacy Policy,4829,32630,f333569a1fc99346c8b2ebe8ce130aaf06eb8df3dccafd...
4,2022-01-13 18:43:08+00:00,476756868644d8f9eeb6746d51418c1964c5ea4c,Update Zoom Privacy Policy,4774,32748,a3470c3441ac72dcfb759bfb8cb547ea92ad2a69b9e7b0...
5,2022-04-05 12:42:05+00:00,60927018fbf523614ab99e5cce0c701e75ca657b,Update Zoom Privacy Policy,4774,32748,d392a862224f89866e88e283f8eb50fa1b076af4f4839a...
6,2022-08-31 00:44:12+00:00,5d41244ff973b31ad590f8a2c68159be1308ecae,Update Zoom Privacy Policy,4775,32753,193561e5e2492418fcb43cd389aa9c7c73552f6fa98179...
7,2022-09-16 18:44:22+00:00,961c91adeeb5859e9e0c99c4467ff17b4b9da348,Update Zoom Privacy Policy,5361,36504,e1818ff782b5e716de749b2dbf91c29a7ea7978105be37...
8,2022-09-20 15:23:57+00:00,0c0014d9036d2bed3f4832efb4206a6063c63fc0,Refilter Zoom Privacy Policy,5358,36369,f7fae2bd2f59c7dade098973e6f545b250e0c2faab9972...
9,2022-11-07 12:46:22+00:00,3fda78b39fabedb62d68a7957d4a60ec085c89fd,Update Zoom Privacy Policy,6080,40957,86aaa00cc0b4bc81d6310ddc5649f60a281a1ca83d0497...


## 4. Build an exact-duplicate audit table

We keep all original Git snapshots for traceability, but later comparisons use only the first chronological occurrence of each exact policy text.


In [6]:
duplicate_audit_df = (
    zoom_versions_df
    .assign(
        is_duplicate_snapshot=zoom_versions_df.duplicated(
            subset="text_hash",
            keep="first",
        )
    )
    .sort_values("timestamp")
    .reset_index(drop=True)
)

# Create timezone-aware empty datetime column
duplicate_audit_df["duplicate_of_timestamp"] = pd.Series(
    pd.NaT,
    index=duplicate_audit_df.index,
    dtype="datetime64[ns, UTC]",
)

duplicate_audit_df["duplicate_of_commit_hash"] = pd.Series(
    pd.NA,
    index=duplicate_audit_df.index,
    dtype="string",
)

first_occurrence = (
    duplicate_audit_df
    .drop_duplicates("text_hash", keep="first")
    .set_index("text_hash")
)

for idx, row in duplicate_audit_df.iterrows():
    if row["is_duplicate_snapshot"]:
        original = first_occurrence.loc[row["text_hash"]]

        duplicate_audit_df.loc[
            idx,
            "duplicate_of_timestamp"
        ] = original["timestamp"]

        duplicate_audit_df.loc[
            idx,
            "duplicate_of_commit_hash"
        ] = original["commit_hash"]

print("Total Git snapshots:", len(duplicate_audit_df))

print(
    "Exact duplicate snapshots:",
    int(duplicate_audit_df["is_duplicate_snapshot"].sum())
)

print(
    "Unique policy texts:",
    int((~duplicate_audit_df["is_duplicate_snapshot"]).sum())
)

display(
    duplicate_audit_df[
        duplicate_audit_df["is_duplicate_snapshot"]
    ][
        [
            "timestamp",
            "commit_hash",
            "commit_message",
            "duplicate_of_timestamp",
            "duplicate_of_commit_hash",
            "word_count",
        ]
    ]
)

Total Git snapshots: 30
Exact duplicate snapshots: 2
Unique policy texts: 28


,timestamp,commit_hash,commit_message,duplicate_of_timestamp,duplicate_of_commit_hash,word_count
15,2023-01-19 06:39:23+00:00,0d97e95b2d0e86853926ce5b694d39f86c905637,Refilter Zoom Privacy Policy,2022-11-19 00:44:15+00:00,48fbc537524305644c9036d37f0eb258ed673f85,6126
16,2023-01-19 06:55:06+00:00,7e19aca26bc91dfaf447f8033b6f15375cb7e36a,Update Zoom Privacy Policy,2022-12-31 00:45:55+00:00,5ffee27416ab3a78e2d3853a8238d191456f3638,6996


## 5. Collapse snapshots into chronological unique policy versions

In [7]:
unique_versions_df = (
    duplicate_audit_df
    .loc[
        ~duplicate_audit_df["is_duplicate_snapshot"]
    ]
    .copy()
    .sort_values("timestamp")
    .reset_index(drop=True)
)

unique_versions_df.insert(
    0,
    "version_id",
    range(1, len(unique_versions_df) + 1),
)

print("Unique policy versions:", len(unique_versions_df))
print(
    "Consecutive unique-version pairs:",
    max(len(unique_versions_df) - 1, 0)
)

display(
    unique_versions_df[
        [
            "version_id",
            "timestamp",
            "commit_hash",
            "commit_message",
            "word_count",
            "char_count",
        ]
    ]
)


Unique policy versions: 28
Consecutive unique-version pairs: 27


,version_id,timestamp,commit_hash,commit_message,word_count,char_count
0,1,2021-08-16 08:55:23+00:00,f5197ca248b099f54512cb518181241af1642749,Start tracking Zoom Privacy Policy,4679,31458
1,2,2021-10-07 21:14:03+00:00,4788e3698c43bc99597d7fd2836b138ef8145e85,Update Zoom Privacy Policy,4705,31710
2,3,2021-11-03 05:10:18+00:00,4fbd6abb19ff42c3a828c95327248576a4966e0a,Refilter Zoom Privacy Policy,4829,32622
3,4,2021-11-23 10:17:24+00:00,7669087fd9438400efc97cc939a9a061588e6da2,Update Zoom Privacy Policy,4829,32630
4,5,2022-01-13 18:43:08+00:00,476756868644d8f9eeb6746d51418c1964c5ea4c,Update Zoom Privacy Policy,4774,32748
5,6,2022-04-05 12:42:05+00:00,60927018fbf523614ab99e5cce0c701e75ca657b,Update Zoom Privacy Policy,4774,32748
6,7,2022-08-31 00:44:12+00:00,5d41244ff973b31ad590f8a2c68159be1308ecae,Update Zoom Privacy Policy,4775,32753
7,8,2022-09-16 18:44:22+00:00,961c91adeeb5859e9e0c99c4467ff17b4b9da348,Update Zoom Privacy Policy,5361,36504
8,9,2022-09-20 15:23:57+00:00,0c0014d9036d2bed3f4832efb4206a6063c63fc0,Refilter Zoom Privacy Policy,5358,36369
9,10,2022-11-07 12:46:22+00:00,3fda78b39fabedb62d68a7957d4a60ec085c89fd,Update Zoom Privacy Policy,6080,40957


## 6. Baseline metric utilities

These metrics intentionally operate on surface text.

They answer questions such as:

- How similar are the two raw texts?
- How much text was added or removed?
- How much did document length change?

They do **not** answer whether the company's permissions expanded.


In [8]:
TOKEN_PATTERN = re.compile(r"\b\w+(?:['’-]\w+)?\b", flags=re.UNICODE)

def tokenize(text):
    return TOKEN_PATTERN.findall(text.lower())

def multiset_difference_counts(old_tokens, new_tokens):
    old_counts = pd.Series(old_tokens).value_counts()
    new_counts = pd.Series(new_tokens).value_counts()

    all_tokens = old_counts.index.union(new_counts.index)

    old_counts = old_counts.reindex(all_tokens, fill_value=0)
    new_counts = new_counts.reindex(all_tokens, fill_value=0)

    added = (new_counts - old_counts).clip(lower=0).sum()
    removed = (old_counts - new_counts).clip(lower=0).sum()

    return int(added), int(removed)

def compute_baseline_metrics(old_text, new_text):
    old_tokens = tokenize(old_text)
    new_tokens = tokenize(new_text)

    added_tokens, removed_tokens = multiset_difference_counts(
        old_tokens,
        new_tokens,
    )

    char_similarity = SequenceMatcher(
        None,
        old_text,
        new_text,
        autojunk=False,
    ).ratio()

    token_similarity = SequenceMatcher(
        None,
        old_tokens,
        new_tokens,
        autojunk=False,
    ).ratio()

    old_word_count = len(old_tokens)
    new_word_count = len(new_tokens)

    old_char_count = len(old_text)
    new_char_count = len(new_text)

    net_word_change = new_word_count - old_word_count
    word_change_pct = (
        (net_word_change / old_word_count) * 100
        if old_word_count else np.nan
    )

    changed_token_volume = added_tokens + removed_tokens
    normalized_change_ratio = (
        changed_token_volume /
        max(old_word_count + new_word_count, 1)
    )

    return {
        "old_word_count": old_word_count,
        "new_word_count": new_word_count,
        "net_word_change": net_word_change,
        "word_change_pct": word_change_pct,
        "old_char_count": old_char_count,
        "new_char_count": new_char_count,
        "net_char_change": new_char_count - old_char_count,
        "char_sequence_similarity": char_similarity,
        "token_sequence_similarity": token_similarity,
        "added_token_count": added_tokens,
        "removed_token_count": removed_tokens,
        "changed_token_volume": changed_token_volume,
        "normalized_change_ratio": normalized_change_ratio,
    }


## 7. Create consecutive unique-version pairs

In [9]:
pair_records = []

for i in range(len(unique_versions_df) - 1):
    old = unique_versions_df.iloc[i]
    new = unique_versions_df.iloc[i + 1]

    metrics = compute_baseline_metrics(
        old["text"],
        new["text"],
    )

    pair_records.append(
        {
            "pair_id": i + 1,
            "company": old["company"],
            "old_version_id": old["version_id"],
            "new_version_id": new["version_id"],
            "old_timestamp": old["timestamp"],
            "new_timestamp": new["timestamp"],
            "days_between_versions": (
                new["timestamp"] - old["timestamp"]
            ).total_seconds() / 86400,
            "old_commit_hash": old["commit_hash"],
            "new_commit_hash": new["commit_hash"],
            "old_commit_message": old["commit_message"],
            "new_commit_message": new["commit_message"],
            "old_text_hash": old["text_hash"],
            "new_text_hash": new["text_hash"],
            **metrics,
            "old_text": old["text"],
            "new_text": new["text"],
        }
    )

version_pairs_df = pd.DataFrame(pair_records)

print("Generated version pairs:", len(version_pairs_df))

display(
    version_pairs_df[
        [
            "pair_id",
            "old_timestamp",
            "new_timestamp",
            "days_between_versions",
            "old_word_count",
            "new_word_count",
            "net_word_change",
            "word_change_pct",
            "char_sequence_similarity",
            "token_sequence_similarity",
            "added_token_count",
            "removed_token_count",
            "normalized_change_ratio",
        ]
    ]
)


Generated version pairs: 27


,pair_id,old_timestamp,new_timestamp,days_between_versions,old_word_count,new_word_count,net_word_change,word_change_pct,char_sequence_similarity,token_sequence_similarity,added_token_count,removed_token_count,normalized_change_ratio
0,1,2021-08-16 08:55:23+00:00,2021-10-07 21:14:03+00:00,52.512963,4685,4718,33,0.704376,0.996011,0.996490,33,0,0.003510
1,2,2021-10-07 21:14:03+00:00,2021-11-03 05:10:18+00:00,26.330729,4718,4851,133,2.818991,0.961015,0.971470,170,37,0.021632
2,3,2021-11-03 05:10:18+00:00,2021-11-23 10:17:24+00:00,20.213264,4851,4851,0,0.000000,0.999479,1.000000,0,0,0.000000
3,4,2021-11-23 10:17:24+00:00,2022-01-13 18:43:08+00:00,51.351204,4851,4832,-19,-0.391672,0.986387,0.985851,58,77,0.013942
4,5,2022-01-13 18:43:08+00:00,2022-04-05 12:42:05+00:00,81.749271,4832,4832,0,0.000000,0.999939,0.999793,1,1,0.000207
5,6,2022-04-05 12:42:05+00:00,2022-08-31 00:44:12+00:00,147.501470,4832,4833,1,0.020695,0.999924,0.999897,1,0,0.000103
6,7,2022-08-31 00:44:12+00:00,2022-09-16 18:44:22+00:00,16.750116,4833,5404,571,11.814608,0.920860,0.919996,623,52,0.065937
7,8,2022-09-16 18:44:22+00:00,2022-09-20 15:23:57+00:00,3.860822,5404,5404,0,0.000000,0.997983,1.000000,0,0,0.000000
8,9,2022-09-20 15:23:57+00:00,2022-11-07 12:46:22+00:00,47.890567,5404,6129,725,13.415988,0.928355,0.923437,734,9,0.064424
9,10,2022-11-07 12:46:22+00:00,2022-11-08 18:44:48+00:00,1.248912,6129,6175,46,0.750530,0.995220,0.994311,55,9,0.005202


## 8. Baseline summary statistics

In [10]:
baseline_summary = (
    version_pairs_df[
        [
            "days_between_versions",
            "net_word_change",
            "word_change_pct",
            "char_sequence_similarity",
            "token_sequence_similarity",
            "added_token_count",
            "removed_token_count",
            "changed_token_volume",
            "normalized_change_ratio",
        ]
    ]
    .describe()
)

display(baseline_summary)


,days_between_versions,net_word_change,word_change_pct,char_sequence_similarity,token_sequence_similarity,added_token_count,removed_token_count,changed_token_volume,normalized_change_ratio
count,27.000000,27.000000,27.000000,27.000000,27.000000,27.000000,27.000000,27.000000,27.000000
mean,40.199704,136.555556,2.262605,0.983122,0.983016,159.111111,22.555556,181.666667,0.014145
std,38.414445,261.398801,4.478969,0.026155,0.026509,270.875835,27.979846,282.814155,0.022583
min,0.499734,-35.000000,-0.430928,0.912622,0.912662,0.000000,0.000000,0.000000,0.000000
25%,11.149300,0.000000,0.000000,0.986381,0.983385,1.000000,1.000000,9.000000,0.000650
50%,26.330729,21.000000,0.256692,0.995776,0.995249,50.000000,9.000000,61.000000,0.004168
75%,54.131580,118.500000,1.583447,0.999144,0.999350,141.000000,36.000000,196.000000,0.013327
max,147.501470,983.000000,15.911298,0.999939,1.000000,1018.000000,82.000000,1053.000000,0.078941


## 9. Rank updates by surface-level change magnitude

In [11]:
largest_changes = (
    version_pairs_df
    .sort_values(
        "normalized_change_ratio",
        ascending=False,
    )
    .reset_index(drop=True)
)

smallest_changes = (
    version_pairs_df
    .sort_values(
        "normalized_change_ratio",
        ascending=True,
    )
    .reset_index(drop=True)
)

print("Largest 10 surface-level changes:")
display(
    largest_changes.head(10)[
        [
            "pair_id",
            "old_timestamp",
            "new_timestamp",
            "old_word_count",
            "new_word_count",
            "net_word_change",
            "added_token_count",
            "removed_token_count",
            "char_sequence_similarity",
            "token_sequence_similarity",
            "normalized_change_ratio",
        ]
    ]
)

print("Smallest 10 surface-level changes:")
display(
    smallest_changes.head(10)[
        [
            "pair_id",
            "old_timestamp",
            "new_timestamp",
            "old_word_count",
            "new_word_count",
            "net_word_change",
            "added_token_count",
            "removed_token_count",
            "char_sequence_similarity",
            "token_sequence_similarity",
            "normalized_change_ratio",
        ]
    ]
)


Largest 10 surface-level changes:


,pair_id,old_timestamp,new_timestamp,old_word_count,new_word_count,net_word_change,added_token_count,removed_token_count,char_sequence_similarity,token_sequence_similarity,normalized_change_ratio
0,13,2022-11-19 00:44:15+00:00,2022-12-30 12:46:18+00:00,6178,7161,983,1018,35,0.912622,0.912662,0.078941
1,7,2022-08-31 00:44:12+00:00,2022-09-16 18:44:22+00:00,4833,5404,571,623,52,0.920860,0.919996,0.065937
2,9,2022-09-20 15:23:57+00:00,2022-11-07 12:46:22+00:00,5404,6129,725,734,9,0.928355,0.923437,0.064424
3,21,2023-09-08 21:57:39+00:00,2023-12-02 11:18:10+00:00,7521,8122,601,681,80,0.945458,0.944448,0.048648
4,2,2021-10-07 21:14:03+00:00,2021-11-03 05:10:18+00:00,4718,4851,133,170,37,0.961015,0.971470,0.021632
5,18,2023-06-30 12:48:00+00:00,2023-08-11 19:15:46+00:00,7337,7520,183,229,46,0.966546,0.965202,0.018510
6,4,2021-11-23 10:17:24+00:00,2022-01-13 18:43:08+00:00,4851,4832,-19,58,77,0.986387,0.985851,0.013942
7,27,2024-07-19 18:19:37+00:00,2024-08-05 18:19:53+00:00,8306,8372,66,139,73,0.986951,0.984411,0.012711
8,24,2023-12-20 00:46:10+00:00,2024-03-01 06:47:40+00:00,8181,8202,21,103,82,0.986376,0.982360,0.011292
9,16,2023-02-24 18:46:12+00:00,2023-04-10 18:45:00+00:00,7162,7298,136,143,7,0.989177,0.989073,0.010373


Smallest 10 surface-level changes:


,pair_id,old_timestamp,new_timestamp,old_word_count,new_word_count,net_word_change,added_token_count,removed_token_count,char_sequence_similarity,token_sequence_similarity,normalized_change_ratio
0,3,2021-11-03 05:10:18+00:00,2021-11-23 10:17:24+00:00,4851,4851,0,0,0,0.999479,1.000000,0.000000
1,8,2022-09-16 18:44:22+00:00,2022-09-20 15:23:57+00:00,5404,5404,0,0,0,0.997983,1.000000,0.000000
2,25,2024-03-01 06:47:40+00:00,2024-03-17 06:48:06+00:00,8202,8202,0,0,0,0.999927,1.000000,0.000000
3,6,2022-04-05 12:42:05+00:00,2022-08-31 00:44:12+00:00,4832,4833,1,1,0,0.999924,0.999897,0.000103
4,19,2023-08-11 19:15:46+00:00,2023-09-05 12:18:54+00:00,7520,7521,1,2,1,0.999148,0.999801,0.000199
5,5,2022-01-13 18:43:08+00:00,2022-04-05 12:42:05+00:00,4832,4832,0,1,1,0.999939,0.999793,0.000207
6,20,2023-09-05 12:18:54+00:00,2023-09-08 21:57:39+00:00,7521,7521,0,4,4,0.999535,0.999468,0.000532
7,14,2022-12-30 12:46:18+00:00,2022-12-31 00:45:55+00:00,7161,7150,-11,0,11,0.999159,0.999231,0.000769
8,11,2022-11-08 18:44:48+00:00,2022-11-10 00:44:41+00:00,6175,6167,-8,1,9,0.999139,0.999190,0.000810
9,12,2022-11-10 00:44:41+00:00,2022-11-19 00:44:15+00:00,6167,6178,11,11,0,0.998546,0.999109,0.000891


## 10. Inspect one pair with a readable unified diff

This is only for human inspection and debugging. It is **not** the SneakyToS semantic model.


In [12]:
PAIR_TO_INSPECT = 1

if PAIR_TO_INSPECT not in set(version_pairs_df["pair_id"]):
    raise ValueError(
        f"PAIR_TO_INSPECT must be between 1 and {len(version_pairs_df)}"
    )

pair = version_pairs_df.loc[
    version_pairs_df["pair_id"] == PAIR_TO_INSPECT
].iloc[0]

old_lines = pair["old_text"].splitlines()
new_lines = pair["new_text"].splitlines()

diff_lines = list(
    unified_diff(
        old_lines,
        new_lines,
        fromfile=f"Zoom {pair['old_timestamp'].date()}",
        tofile=f"Zoom {pair['new_timestamp'].date()}",
        lineterm="",
        n=2,
    )
)

print(
    f"Pair {PAIR_TO_INSPECT}: "
    f"{pair['old_timestamp'].date()} → "
    f"{pair['new_timestamp'].date()}"
)
print()

preview_limit = 300

if len(diff_lines) > preview_limit:
    print(
        f"Showing first {preview_limit} of "
        f"{len(diff_lines)} diff lines."
    )
    print()
    diff_lines = diff_lines[:preview_limit]

print("\n".join(diff_lines))


Pair 1: 2021-08-16 → 2021-10-07

--- Zoom 2021-08-16
+++ Zoom 2021-10-07
@@ -102,4 +102,6 @@
 Zoom does not allow children under the age of 16 to sign up for a Zoom account.  
 
+For educational organizations that use Zoom Products to provide educational services to children under 18 using Zoom Products, Zoom’s Children’s Educational Privacy Statement is available [here](https://explore.zoom.us/en/schools-privacy-statement/).
+
 How to Contact Us


## 11. Save baseline datasets

In [13]:
unique_versions_path = (
    PROCESSED_PATH
    / "zoom_unique_policy_versions.pkl"
)

duplicate_audit_path = (
    PROCESSED_PATH
    / "zoom_duplicate_snapshot_audit.csv"
)

version_pairs_path = (
    PROCESSED_PATH
    / "zoom_version_pairs_baseline.pkl"
)

version_pairs_summary_path = (
    PROCESSED_PATH
    / "zoom_version_pairs_baseline.csv"
)

unique_versions_df.to_pickle(unique_versions_path)

duplicate_audit_df.to_csv(
    duplicate_audit_path,
    index=False,
)

version_pairs_df.to_pickle(
    version_pairs_path,
)

version_pairs_df.drop(
    columns=["old_text", "new_text"]
).to_csv(
    version_pairs_summary_path,
    index=False,
)

print("Saved:", unique_versions_path)
print("Saved:", duplicate_audit_path)
print("Saved:", version_pairs_path)
print("Saved:", version_pairs_summary_path)


Saved: c:\Users\arjun\Desktop\SneakyToS\data\processed\zoom_unique_policy_versions.pkl
Saved: c:\Users\arjun\Desktop\SneakyToS\data\processed\zoom_duplicate_snapshot_audit.csv
Saved: c:\Users\arjun\Desktop\SneakyToS\data\processed\zoom_version_pairs_baseline.pkl
Saved: c:\Users\arjun\Desktop\SneakyToS\data\processed\zoom_version_pairs_baseline.csv


# Notebook 02 Output

After successful execution, this notebook produces:

- `data/processed/zoom_unique_policy_versions.pkl`
- `data/processed/zoom_duplicate_snapshot_audit.csv`
- `data/processed/zoom_version_pairs_baseline.pkl`
- `data/processed/zoom_version_pairs_baseline.csv`

## What we have now

For Zoom, we now have a clean sequence of unique policy versions and consecutive version pairs with surface-level change measurements.

## Next stage

Notebook 03 will move from **whole-document comparison** to **document structure**:

1. Parse Markdown structure.
2. Segment policies into meaningful sections/clauses.
3. Preserve section context and version metadata.
4. Prepare clause units for semantic alignment.

That is where the NLP pipeline starts becoming clause-level.
